# Data Mining and Regression Dashboard

Reproducible analysis of product-defect records using OLAP summaries regression PCA KMeans clustering and frequent-pattern mining.

In [ ]:
import matplotlib.pyplot as plt
from analysis import (cluster_projection, evaluate_models, load_dataset, mine_frequent_patterns, olap_summary, prepare_features)

In [ ]:
df = load_dataset()
X, y = prepare_features(df)
print(f'Rows: {len(df):,} | Encoded features: {X.shape[1]}')
df.head()

## OLAP-style aggregation

In [ ]:
summary = olap_summary(df)
display(summary)
summary['average_cost'].plot(kind='bar', title='Average repair cost by severity', ylabel='Repair cost (₹)')
plt.tight_layout()

## Regression evaluation

Models are compared against a mean baseline on the same held-out split. KNN scaling and neighbour selection occur inside a cross-validated pipeline.

In [ ]:
metrics, knn_curve, best_k = evaluate_models(X, y)
display(metrics.style.format({'r2': '{:.3f}', 'mae': '₹{:.2f}', 'rmse': '₹{:.2f}'}))
print('Selected KNN neighbours:', best_k)
metrics.plot.bar(x='model', y='r2', legend=False, title='Held-out R² by model')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()

## PCA and KMeans

In [ ]:
projection, clusters, explained = cluster_projection(X)
plt.scatter(projection[:, 0], projection[:, 1], c=clusters, s=18, alpha=0.7, cmap='viridis')
plt.title(f'KMeans clusters in PCA space ({explained.sum():.1%} variance shown)')
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.tight_layout()

## Frequent itemsets

Items are namespaced so a severity label cannot collide with a defect-type label. Transactions group observed characteristics by product.

In [ ]:
patterns = mine_frequent_patterns(df, min_support=0.3)
display(patterns.head(15))

## Interpretation

Negative or near-zero held-out R² indicates that the available attributes do not explain repair cost better than the mean baseline. This is a valid analytical finding and signals that richer cost drivers are needed.